In [25]:
from datetime import datetime
import os 
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:

    pass

PASSWORD_EDU = os.getenv("EDUPAGE_PASSWORD", "")

if PASSWORD_EDU is not None:
    PASSWORD_EDU = PASSWORD_EDU.strip()


if not PASSWORD_EDU:
    raise SystemExit("PASSWORD not found in environment. Add it to your .env (PASSWORD=...) or set it in the OS environment.")
from edupage_api import Edupage
from edupage_api.timeline import EventType

edupage = Edupage()
edupage.login(
    "KristianSiska",
    PASSWORD_EDU,
    "spsezoska",
)

notifications = edupage.get_notifications()

homework_not_due = 0
@dataclass
class Homework:
    event_id: int
    timestamp: datetime
    text: str
    author: Optional[str] = None
    recipient: Optional[str] = None
    event_type: Optional[Any] = None
    additional_data: Dict[str, Any] = field(default_factory=dict)

    # Common additional_data fields
    predmetid: Optional[str] = None
    triedaid: Optional[str] = None
    skupinyids: List[str] = field(default_factory=list)
    nazov: Optional[str] = None
    popis: Optional[str] = None
    date: Optional[str] = None
    due_date: Optional[datetime] = None
    typ: Optional[Any] = None
    id: Optional[str] = None
    skupiny: List[str] = field(default_factory=list)
    zmena: Optional[bool] = None
    oldVals: Optional[Dict[str, Any]] = None
    superid: Optional[str] = None
    etestCards: Optional[int] = None
    attachements: Optional[Any] = None
    planid: Optional[str] = None

    # Other timeline flags
    is_done: Optional[bool] = None
    done_at: Optional[datetime] = None
    is_starred: Optional[bool] = None
    reaction_count: Optional[int] = None
    created_at: Optional[datetime] = None
    is_removed: Optional[bool] = None

    @classmethod
    def from_timeline_event(cls, event) -> "Homework":
        ad = getattr(event, "additional_data", {}) or {}
        old_vals = ad.get("oldVals")

        # prefer date from additional_data, fall back to oldVals when present
        date_str = ad.get("date")
        if not date_str and isinstance(old_vals, dict):
            date_str = old_vals.get("date")

        due_date = None
        if date_str:
            try:
                due_date = datetime.strptime(date_str, "%Y-%m-%d")
            except Exception:
                due_date = None

        nazov = ad.get("nazov") or ad.get("title")

        return cls(
            event_id=getattr(event, "event_id", None),
            timestamp=getattr(event, "timestamp", None),
            text=getattr(event, "text", ""),
            author=getattr(event, "author", None),
            recipient=getattr(event, "recipient", None) or ad.get("recipient"),
            event_type=getattr(event, "event_type", None),
            additional_data=ad,
            predmetid=ad.get("predmetid"),
            triedaid=ad.get("triedaid"),
            skupinyids=ad.get("skupinyids") or [],
            nazov=nazov,
            popis=ad.get("popis"),
            date=date_str,
            due_date=due_date,
            typ=ad.get("typ"),
            id=ad.get("id"),
            skupiny=ad.get("skupiny") or [],
            zmena=ad.get("zmena"),
            oldVals=old_vals if isinstance(old_vals, dict) else None,
            superid=ad.get("superid"),
            etestCards=ad.get("etestCards"),
            attachements=ad.get("attachements") or ad.get("attachments"),
            planid=ad.get("planid"),
            is_done=getattr(event, "is_done", None),
            done_at=getattr(event, "done_at", None),
            is_starred=getattr(event, "is_starred", None),
            reaction_count=getattr(event, "reaction_count", None),
            created_at=getattr(event, "created_at", None),
            is_removed=getattr(event, "is_removed", None),
        )


homework = list(filter(lambda x: x.event_type == EventType.HOMEWORK, notifications))
print(homework, file=open("homework_debug.txt", "w", encoding="utf-8"))
for ev in homework:
    hw = Homework.from_timeline_event(ev)

    hw_title = hw.nazov
    hw_class = hw.recipient

    text = f"Homework from {hw.timestamp}\n"
    hw_text = hw_title or hw.text.replace("\n", " ")
    text += f"{hw_text}\n{hw_class}\n"

    if hw.due_date:
        now = datetime.now()
        text += f"\nDue to: {hw.due_date} -> "

        if hw.due_date > now:
            text += "DUE"
        else:
            homework_not_due += 1
            text += "UNFINISHED"

    print(text)
    print("—")

print(f"You have {homework_not_due} unfinished homework assignments.")





Homework from 2026-06-15 14:42:05
Zmenené: opakovanie na koncorocnu písomku: modalne slovesa: koennen, wollen, minuly cas : perfekt slovies, prateritum slovies haben&sein, Berufe,Hobbies 
II.D - 2. Skupina · Nemecký jazyk

Due to: 2026-06-16 00:00:00 -> DUE
—
Homework from 2026-06-10 10:40:13
Písomka PowerPoint
II.D - 2. Skupina · Aplikovaná informatika

Due to: 2026-06-10 00:00:00 -> UNFINISHED
—
Homework from 2026-06-05 08:32:28
Command list
II.D - 2. Skupina · Databázové systémy

Due to: 2026-06-12 00:00:00 -> UNFINISHED
—
Homework from 2026-06-05 07:59:00
Zmenené: 19. zadanie
II.D - 2. Skupina · Databázové systémy

Due to: 2026-06-11 00:00:00 -> UNFINISHED
—
Homework from 2026-06-03 12:53:50
Quiz next week - See Teams
II.D · Analýza dát

Due to: 2026-06-10 00:00:00 -> UNFINISHED
—
Homework from 2026-06-02 11:01:09
Formular
II.D - 2. Skupina · Skriptovacie jazyky

Due to: 2026-06-03 00:00:00 -> UNFINISHED
—
Homework from 2026-06-01 10:53:04
Komplexný dizajn - OneDrive 13 - zadanie 3

## 🍽️ Test: Sign up for lunch

Uses `edupage.get_meals(date)` to read the menu, then `lunch.choose(edupage, n)` to order menu *n* (or `lunch.sign_off(edupage)` to cancel). Ordering is guarded behind `DO_ORDER` so running the cell doesn't accidentally place a real order.

In [26]:
# Sign up for lunch: get_meals(date) -> meals.lunch.choose(edupage, n) / sign_off(edupage)
from datetime import date, timedelta

# get_meals() needs a real datetime.date (it calls .strftime), not a string.
target = date.today() + timedelta(days=1)   # the NEXT day, not today
print(f"Canteen for: {target}")
meals = edupage.get_meals(target)

if meals is None:
    print(f"No meals found for {target}")
else:
    lunch = meals.lunch
    if lunch is None:
        print(f"No lunch available for {target}")
    else:
        print(f"Lunch:      {lunch.title}")
        print(f"Changeable until: {lunch.can_be_changed_until}")
        print(f"Currently ordered: {lunch.ordered_meal}")
        print("Available menus:")
        for menu in lunch.menus:
            print(f"   {menu.number}. {menu.name}  ({menu.weight})  | allergens: {menu.allergens}")

        # --- Place the order ---
        # choose() actually submits to EduPage, so it's guarded by a flag.
        # Set DO_ORDER = True to really sign up; choice_number is the menu number (1-8).
        DO_ORDER = True
        choice_number = 1
        if DO_ORDER:
            lunch.choose(edupage, choice_number)
            print(f"\n✅ Ordered menu #{choice_number}. ordered_meal is now: {lunch.ordered_meal}")
        else:
            print(f"\n[dry run] Would order menu #{choice_number}. Set DO_ORDER=True to submit.")
            print("To cancel an existing order instead: lunch.sign_off(edupage)")


Canteen for: 2026-06-16
Lunch:       Polievka tekvicová na kyslo 0,33l
A:  Pečené kuracie stehno na tymiáne(240g), zemiaky(200g), kompót
B:  Hubové rizoto, strúhaný syr (400g)
C:  Cézar šalát, dresing(350g)

Changeable until: 2026-06-15 18:00
Currently ordered: A
Available menus:
   None.  Polievka tekvicová na kyslo 0,33l  (None)  | allergens: 1, 7, 9
   A.  Pečené kuracie stehno na tymiáne(240g), zemiaky(200g), kompót  (None)  | allergens: 9
   B.  Hubové rizoto, strúhaný syr (400g)  (None)  | allergens: 7, 9
   C.  Cézar šalát, dresing(350g)  (None)  | allergens: 1, 4, 7

✅ Ordered menu #1. ordered_meal is now: A


## 📚 Test: Homework detailed info + files

Reuses the `Homework` dataclass defined above to print full detail per homework (subject, author, due date, done state, IDs/superid) and lists any attached files from `additional_data['attachements']`.

In [27]:
# Detailed homework info + any attached files.
# The Homework dataclass (defined above) unpacks the timeline event's additional_data.
homework_events = [e for e in notifications if e.event_type == EventType.HOMEWORK]
print(f"Found {len(homework_events)} homework events\n")


def attachment_url(path: str) -> str:
    """Turn an EduPage attachment path into a full URL."""
    if not path:
        return path
    if path.startswith("http"):
        return path
    return f"https://{edupage.subdomain}.edupage.org/{path.lstrip('/')}"


for ev in homework_events:
    hw = Homework.from_timeline_event(ev)
    print("=" * 70)
    print(f"Title:       {hw.nazov or (hw.text or '').replace(chr(10), ' ')}")
    print(f"Subject:     {hw.recipient}")
    print(f"Author:      {hw.author}")
    print(f"Posted:      {hw.timestamp}")
    print(f"Due date:    {hw.due_date}")
    print(f"Done:        {hw.is_done}  (at {hw.done_at})")
    print(f"IDs:         superid={hw.superid}  id={hw.id}  planid={hw.planid}  predmetid={hw.predmetid}")
    print(f"e-test cards: {hw.etestCards}")
    if hw.popis:
        print(f"Description: {hw.popis}")

    # Files / attachments. EduPage stores them in additional_data['attachements']
    # as { 'filename': 'cloud_path' } — present only when files were attached.
    attachments = hw.attachements
    if attachments:
        print("Attachments:")
        if isinstance(attachments, dict):
            for name, path in attachments.items():
                print(f"   - {name}: {attachment_url(path)}")
        else:
            print(f"   {attachments}")
    else:
        print("Attachments: none in timeline data"
              + (f" (this homework links an e-test, superid={hw.superid})" if hw.etestCards else ""))

print("=" * 70)


Found 12 homework events

Title:       Zmenené: opakovanie na koncorocnu písomku: modalne slovesa: koennen, wollen, minuly cas : perfekt slovies, prateritum slovies haben&sein, Berufe,Hobbies 
Subject:     II.D - 2. Skupina · Nemecký jazyk
Author:      Janka Morongová
Posted:      2026-06-15 14:42:05
Due date:    2026-06-16 00:00:00
Done:        False  (at None)
IDs:         superid=36627  id=ACE746A85DA0834F7DE1  planid=6704  predmetid=35174
e-test cards: 0
Attachments: none in timeline data
Title:       Písomka PowerPoint
Subject:     II.D - 2. Skupina · Aplikovaná informatika
Author:      Lenka Korenková
Posted:      2026-06-10 10:40:13
Due date:    2026-06-10 00:00:00
Done:        True  (at 2026-06-03 11:11:50)
IDs:         superid=36390  id=0EC2F72C0C1FDA30B6A0  planid=6551  predmetid=51526
e-test cards: 2
Attachments: none in timeline data (this homework links an e-test, superid=36390)
Title:       Command list
Subject:     II.D - 2. Skupina · Databázové systémy
Author:      Adam

## 📎 Test: Get files attached to an e-test homework

The timeline only says *"links an e-test"* — the actual files live in the e-test material. We POST the homework's `superid` to `/elearning/?cmd=EtestCreator&akcia=getResultsData` (body encoded as `eqap=base64(...)&eqaz=0`), then dig into `materialData.cardsData[*].content` where `FileETestWidget` widgets hold `props.files[]` (each has a `name` + a downloadable `src`). Set `DOWNLOAD = True` to save them locally.

*(Verified: `superid=36216` returns two images, e.g. a 982 KB PNG.)*

In [28]:
import base64
import json
import os
import urllib.parse


def _edu_encode_body(data: dict) -> str:
    """EduPage expects the POST body as eqap=<urlencode(base64(querystring))>&eqaz=0."""
    query = urllib.parse.urlencode(data)
    b64 = base64.b64encode(query.encode("utf-8")).decode("ascii")
    return f"eqap={urllib.parse.quote(b64)}&eqaz=0"


def fetch_etest_data(superid) -> dict:
    """Fetch the full e-test/material payload for a homework's superid.

    Hits /elearning/?cmd=EtestCreator&akcia=getResultsData, which returns
    {viewMode, resultsData, materialData, ...}. The card content (and the files
    attached to it) live under materialData.cardsData[<cardid>].content.
    """
    url = (f"https://{edupage.subdomain}.edupage.org"
           f"/elearning/?cmd=EtestCreator&akcia=getResultsData")
    headers = {
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "X-Requested-With": "XMLHttpRequest",
        "Referer": f"https://{edupage.subdomain}.edupage.org/",
    }
    resp = edupage.session.post(url, data=_edu_encode_body({"superid": str(superid)}), headers=headers)
    resp.raise_for_status()
    return resp.json()


def _collect_files(node, out):
    """Recursively pull every {name, src} file out of e-test card widgets
    (FileETestWidget / ImageETestWidget store them in props.files[])."""
    if isinstance(node, dict):
        files = node.get("props", {}).get("files") if isinstance(node.get("props"), dict) else None
        if isinstance(files, list):
            for f in files:
                if isinstance(f, dict) and f.get("src"):
                    out.append({
                        "name": f.get("name") or f.get("src").rsplit("/", 1)[-1],
                        "url": f"https://{edupage.subdomain}.edupage.org{f['src']}",
                        "type": f.get("type"),
                        "extension": f.get("extension"),
                    })
        for v in node.values():
            _collect_files(v, out)
    elif isinstance(node, list):
        for v in node:
            _collect_files(v, out)


def get_etest_files(superid) -> list:
    """Return a list of files attached to the e-test cards of a homework."""
    data = fetch_etest_data(superid)
    cards = data.get("materialData", {}).get("cardsData", {})
    out = []
    for card in cards.values():
        content = card.get("content")
        if isinstance(content, str):
            try:
                content = json.loads(content)
            except json.JSONDecodeError:
                continue
        _collect_files(content, out)
    # de-duplicate by url
    seen, unique = set(), []
    for f in out:
        if f["url"] not in seen:
            seen.add(f["url"])
            unique.append(f)
    return unique


# --- List files for every homework that links an e-test ---
DOWNLOAD = False                     # set True to actually save the files
DOWNLOAD_DIR = "homework_files"

for ev in homework_events:
    hw = Homework.from_timeline_event(ev)
    if not hw.etestCards or not hw.superid:
        continue
    files = get_etest_files(hw.superid)
    print("=" * 70)
    print(f"{hw.nazov or hw.text}  (superid={hw.superid})")
    if not files:
        print("   no files in e-test cards")
        continue
    for f in files:
        print(f"   📎 {f['name']}  [{f['type']}/{f['extension']}]")
        print(f"      {f['url']}")
        if DOWNLOAD:
            os.makedirs(DOWNLOAD_DIR, exist_ok=True)
            dest = os.path.join(DOWNLOAD_DIR, f"{hw.superid}_{f['name']}")
            r = edupage.session.get(f["url"])
            r.raise_for_status()
            with open(dest, "wb") as fh:
                fh.write(r.content)
            print(f"      ↳ saved {len(r.content):,} bytes to {dest}")


Písomka PowerPoint  (superid=36390)
   📎 Písomka B - Vývoj hier - Od nápadu k hitu.pptx  [/None]
      https://spsezoska.edupage.org/elearning/ruqjzfpv?z%3An1l5ZTc6ogPGCWkHfCpiSUK%2Fj%2BBmtEmUDVTUvz5aIAaG1NMYv%2FzBlKN3v8CaNUi7hcUywz8gj%2BL2J0iPZAlDSA%3D%3D
   📎 Písomka A - Sociálne siete a ich vplyv.pptx  [/None]
      https://spsezoska.edupage.org/elearning/ruqjzfpv?z%3AxecdTMVQlUTUOuzHgawpVwyN76Z2l5Bsw%2BSIoueQCf0xlDYjUANb%2F8NrJsEMtRQYtGkXS2xDnp23qPG1OZ5CDw%3D%3D
Command list  (superid=36438)
   no files in e-test cards
Zmenené: 19. zadanie  (superid=36435)
   no files in e-test cards
Formular  (superid=36362)
   no files in e-test cards
Dobrovoľná DÚ (PowerPoint)  (superid=36296)
   📎 Dobrovoľná úloha - PowerPoint.docx  [/None]
      https://spsezoska.edupage.org/elearning/ruqjzfpv?z%3Ak3T8qoZwr55ugVM1N%2BXA29qfQOSEmQjdkPX8UtwQtYYV3BnCpmLgaMdOPt1rX3opsPIjHXtRa3RKUyZeed0Img%3D%3D
   📎 DÚ - Ako funguje AI chatbot.pptx  [/None]
      https://spsezoska.edupage.org/elearning/ruqjzfpv?z%

## 🎓 Test: Exact grades from all subjects

Uses `edupage.get_grades()` (all available `EduGrade`s) and groups them by `subject_name`. Each grade shows the value, percentage/max points, class average, and title.

In [29]:
# get_grades() returns every available EduGrade; we group them by subject.
from collections import defaultdict

grades = edupage.get_grades()
print(f"Total grades: {len(grades)}\n")

by_subject = defaultdict(list)
for g in grades:
    by_subject[g.subject_name or f"subject#{g.subject_id}"].append(g)

for subject in sorted(by_subject):
    subject_grades = sorted(by_subject[subject], key=lambda x: x.date)
    print(f"=== {subject}  ({len(subject_grades)} grades) ===")
    for g in subject_grades:
        line = f"  {g.date:%Y-%m-%d}  grade={str(g.grade_n):>5}"
        if g.max_points:
            line += f"  ({g.percent:.0f}%, max {g.max_points})"
        if g.class_grade_avg is not None:
            line += f"  [class avg: {g.class_grade_avg}]"
        line += f"  — {g.title}"
        print(line)
    print()


Total grades: 59

=== ANJ  (3 grades) ===
  2026-03-05  grade=  1.0  [class avg: 1.08]  — My holidays
  2026-03-19  grade=  1.0  [class avg: 1.54]  — Ind.Speech
  2026-04-30  grade=  1.0  [class avg: 1.2]  — Passive V.

=== API  (2 grades) ===
  2026-04-10  grade=21.75  (109%, max 20.0)  [class avg: 18.08]  — Písomka Excel
  2026-06-04  grade= 22.0  (110%, max 20.0)  [class avg: 20.69]  — Písomka PowerPoint

=== DSY  (7 grades) ===
  2026-02-27  grade=  1.0  [class avg: 1.0]  — 12.optimalizácia výkonu DB
  2026-02-27  grade=  1.0  [class avg: 1.36]  — 11. Zmena štruktúry, transakcie a pohľady 
  2026-05-29  grade=  1.0  [class avg: 1.0]  — 15. Stored Procedures
  2026-05-29  grade=  1.0  [class avg: 1.0]  — 16. Rutiny v phpMyAdmin 
  2026-05-29  grade=  1.0  [class avg: 1.0]  — 17.Fulltextové vyhľadávanie 
  2026-05-29  grade=  1.0  [class avg: 1.36]  — 18. Cudzie kľúče - 1.časť 
  2026-06-05  grade=  1.0  [class avg: 1.0]  — Test - 20. Reštrikcie v SQL DB - ON DELETE a ON UPDATE 

===

## 🕒 Test: Attendance

EduPage's *Dochádzka* isn't exposed by a dedicated `edupage-api` method. Attendance surfaces only as timeline events — we filter the notification history for arrival / absence / excuse event types.

In [30]:
# Attendance is not a first-class module in edupage-api — it surfaces as timeline events.
# Relevant EventTypes: pipnutie (arrival), ospravedlnenka (excused), student_absent, representation.
from datetime import date, timedelta

attendance_types = {
    EventType.ARRIVAL_TO_SCHOOL: "Arrival (card tap)",
    EventType.EXCUSED_LESSON:    "Excused absence",
    EventType.STUDENT_ABSENT:    "Absent",
    EventType.REPRESENTATION:    "Representation",
}

# Pull a wider window so attendance events have a chance to show up
try:
    history = edupage.get_notification_history(date.today() - timedelta(days=30))
except Exception as e:
    print(f"History fetch failed ({e}); falling back to current notifications")
    history = notifications

attendance = [e for e in history if e.event_type in attendance_types]
print(f"Found {len(attendance)} attendance-related events in the last 30 days\n")

for e in sorted(attendance, key=lambda x: x.timestamp):
    label = attendance_types[e.event_type]
    text = (e.text or "").replace("\n", " ").strip()
    print(f"{e.timestamp:%Y-%m-%d %H:%M}  [{label}]  {text}")

if not attendance:
    print("No attendance events found in the window.")
    print("EduPage 'Dochádzka' has no dedicated edupage-api method — only the timeline")
    print("event types above expose attendance, and only if your school records them.")


Found 21 attendance-related events in the last 30 days

2026-05-26 08:20  [Absent]  Absence 26.05.2026
2026-05-28 08:27  [Absent]  Absence 28.05.2026
2026-05-29 09:39  [Excused absence]  Nová ospravedlnenka 31.03.2026
2026-06-02 10:59  [Absent]  Absence 02.06.2026
2026-06-03 08:15  [Absent]  Absence 03.06.2026
2026-06-04 08:44  [Absent]  Absence 04.06.2026
2026-06-08 21:18  [Excused absence]  Nová ospravedlnenka 12.03.2026
2026-06-08 21:18  [Excused absence]  Nová ospravedlnenka 25.02.2026
2026-06-08 21:18  [Excused absence]  Nová ospravedlnenka 10.04.2026
2026-06-08 21:18  [Excused absence]  Nová ospravedlnenka 28.05.2026
2026-06-08 21:19  [Excused absence]  Nová ospravedlnenka 03.06.2026
2026-06-08 21:19  [Excused absence]  Nová ospravedlnenka 04.06.2026
2026-06-08 21:19  [Excused absence]  Nová ospravedlnenka 26.05.2026
2026-06-08 21:19  [Excused absence]  Nová ospravedlnenka 02.06.2026
2026-06-09 12:38  [Absent]  Absence 09.06.2026
2026-06-10 12:46  [Absent]  Absence 10.06.2026
202